In [2]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Port Lavaca Jul 2023.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_July_2023.m21fm - Result Files\Simulated_Aransas.dfs0"
sim_item_index = 2   # Item no. 3 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2023-07-01 02:26:50")
t_end   = pd.to_datetime("2023-07-16 19:46:27")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 3 (0-based index=2)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# --- Compute amplitudes (half of peak-to-trough range) in the window ---
amp_meas = 0.5 * (df["Measured"].max() - df["Measured"].min())
amp_simu = 0.5 * (df["Simulated"].max() - df["Simulated"].min())
amp_ratio = amp_simu / amp_meas if amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_sec = np.median(np.diff(df.index.view("int64"))) / 1e9  # ns -> s
if not np.isfinite(dt_sec) or dt_sec <= 0:
    # Fallback to 10 minutes if needed
    dt_sec = 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    # corr(Measured, Simulated shifted by k samples)
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

# Interpretation:
# If we correlate Measured vs Sim.shift(k), a positive k means SIM is shifted forward (later) to best match Measured,
# i.e., SIM lags OBS by (k * dt_hours). This is the desired phase difference (Sim - Obs).
phase_diff_hours = best_k * dt_hours

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print(f"Measured amplitude  : {amp_meas:.3f} m")
print(f"Simulated amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2023-07-01 02:26:50  to  2023-07-16 19:46:27
Measured amplitude  : 0.259 m
Simulated amplitude : 0.325 m
Amplitude ratio (Sim/Obs): 1.253
Phase difference (Sim - Obs): 2.00 hours
Max-correlation at lag k=4 samples (corr=0.900); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\3208332051.py:52: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [5]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Portocornor JUL 2023.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_July_2023.m21fm - Result Files\Simulated_Aransas.dfs0"
sim_item_index = 1   # Item no. 2 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2023-07-01 13:35:32")
t_end   = pd.to_datetime("2023-07-16 12:03:10")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 2 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# --- Compute amplitudes (half of peak-to-trough range) in the window ---
amp_meas = 0.5 * (df["Measured"].max() - df["Measured"].min())
amp_simu = 0.5 * (df["Simulated"].max() - df["Simulated"].min())
amp_ratio = amp_simu / amp_meas if amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_sec = np.median(np.diff(df.index.view("int64"))) / 1e9  # ns -> s
if not np.isfinite(dt_sec) or dt_sec <= 0:
    # Fallback to 10 minutes if needed
    dt_sec = 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    # corr(Measured, Simulated shifted by k samples)
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

# Interpretation:
# If we correlate Measured vs Sim.shift(k), a positive k means SIM is shifted forward (later) to best match Measured,
# i.e., SIM lags OBS by (k * dt_hours). This is the desired phase difference (Sim - Obs).
phase_diff_hours = best_k * dt_hours

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print(f"Measured amplitude  : {amp_meas:.3f} m")
print(f"Simulated amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2023-07-01 13:35:32  to  2023-07-16 12:03:10
Measured amplitude  : 0.238 m
Simulated amplitude : 0.278 m
Amplitude ratio (Sim/Obs): 1.165
Phase difference (Sim - Obs): 0.50 hours
Max-correlation at lag k=1 samples (corr=0.915); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\36997928.py:52: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [6]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\La quinta channel July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\Simulated_Matagorda v2.dfs0"
sim_item_index = 7   # Item no. 8 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-08 1:16:20")
t_end   = pd.to_datetime("2022-07-29 20:24:38")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 8 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# --- Compute amplitudes (half of peak-to-trough range) in the window ---
amp_meas = 0.5 * (df["Measured"].max() - df["Measured"].min())
amp_simu = 0.5 * (df["Simulated"].max() - df["Simulated"].min())
amp_ratio = amp_simu / amp_meas if amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_sec = np.median(np.diff(df.index.view("int64"))) / 1e9  # ns -> s
if not np.isfinite(dt_sec) or dt_sec <= 0:
    # Fallback to 10 minutes if needed
    dt_sec = 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    # corr(Measured, Simulated shifted by k samples)
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

# Interpretation:
# If we correlate Measured vs Sim.shift(k), a positive k means SIM is shifted forward (later) to best match Measured,
# i.e., SIM lags OBS by (k * dt_hours). This is the desired phase difference (Sim - Obs).
phase_diff_hours = best_k * dt_hours

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print(f"Measured amplitude  : {amp_meas:.3f} m")
print(f"Simulated amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2022-07-08 01:16:20  to  2022-07-29 20:24:38
Measured amplitude  : 0.223 m
Simulated amplitude : 0.199 m
Amplitude ratio (Sim/Obs): 0.894
Phase difference (Sim - Obs): 2.00 hours
Max-correlation at lag k=4 samples (corr=0.881); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\3042689531.py:52: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [7]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\USS Lexington July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\Simulated_Matagorda v2.dfs0"
sim_item_index = 6   # Item no. 7 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 12:43:18")
t_end   = pd.to_datetime("2022-07-31 9:26:50")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 7 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# --- Compute amplitudes (half of peak-to-trough range) in the window ---
amp_meas = 0.5 * (df["Measured"].max() - df["Measured"].min())
amp_simu = 0.5 * (df["Simulated"].max() - df["Simulated"].min())
amp_ratio = amp_simu / amp_meas if amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_sec = np.median(np.diff(df.index.view("int64"))) / 1e9  # ns -> s
if not np.isfinite(dt_sec) or dt_sec <= 0:
    # Fallback to 10 minutes if needed
    dt_sec = 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    # corr(Measured, Simulated shifted by k samples)
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

# Interpretation:
# If we correlate Measured vs Sim.shift(k), a positive k means SIM is shifted forward (later) to best match Measured,
# i.e., SIM lags OBS by (k * dt_hours). This is the desired phase difference (Sim - Obs).
phase_diff_hours = best_k * dt_hours

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print(f"Measured amplitude  : {amp_meas:.3f} m")
print(f"Simulated amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2022-07-06 12:43:18  to  2022-07-31 09:26:50
Measured amplitude  : 0.242 m
Simulated amplitude : 0.205 m
Amplitude ratio (Sim/Obs): 0.847
Phase difference (Sim - Obs): 27.50 hours
Max-correlation at lag k=55 samples (corr=0.841); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\263266961.py:52: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [10]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Port Lavaca Aug 2017 Harvey.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_Hurricane_Harvey.m21fm - Result Files\Simulated port lavaca3.dfs0"
sim_item_index = 0   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2017-08-23 17:35:49")
t_end   = pd.to_datetime("2017-08-28 12:35:49")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 1 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# --- Compute amplitudes (half of peak-to-trough range) in the window ---
amp_meas = 0.5 * (df["Measured"].max() - df["Measured"].min())
amp_simu = 0.5 * (df["Simulated"].max() - df["Simulated"].min())
amp_ratio = amp_simu / amp_meas if amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_sec = np.median(np.diff(df.index.view("int64"))) / 1e9  # ns -> s
if not np.isfinite(dt_sec) or dt_sec <= 0:
    # Fallback to 10 minutes if needed
    dt_sec = 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    # corr(Measured, Simulated shifted by k samples)
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

# Interpretation:
# If we correlate Measured vs Sim.shift(k), a positive k means SIM is shifted forward (later) to best match Measured,
# i.e., SIM lags OBS by (k * dt_hours). This is the desired phase difference (Sim - Obs).
phase_diff_hours = best_k * dt_hours

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print(f"Measured amplitude  : {amp_meas:.3f} m")
print(f"Simulated amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2017-08-23 17:35:49  to  2017-08-28 12:35:49
Measured amplitude  : 1.120 m
Simulated amplitude : 1.497 m
Amplitude ratio (Sim/Obs): 1.336
Phase difference (Sim - Obs): 1.70 hours
Max-correlation at lag k=17 samples (corr=0.933); dt = 0.100 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\441204117.py:52: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [11]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Port Lavaca Aug 2017 Harvey.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_Hurricane_Harvey.m21fm - Result Files\Simulated port lavaca3.dfs0"
sim_item_index = 0   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2017-08-23 17:35:49")
t_end   = pd.to_datetime("2017-08-28 12:35:49")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 1 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# ---------- Helper: maximum amplitude (largest adjacent peak↔trough swing / 2) ----------
def max_half_range_amplitude(series: pd.Series):
    """
    Returns:
        amp_max: maximum half-range amplitude (float)
        t1, t2 : timestamps of the trough/peak pair defining amp_max (pd.Timestamp)
        v1, v2 : values at those timestamps (float, float)
    Notes:
        Finds local extrema (including endpoints) and computes the maximum
        adjacent peak-to-trough half-range within the window.
    """
    s = series.dropna()
    if len(s) < 3:
        # Fallback: use global half-range
        vmin, vmax = s.min(), s.max()
        return 0.5 * (vmax - vmin), s.idxmin(), s.idxmax(), vmin, vmax

    vals = s.values
    idxs = s.index

    # First differences
    d = np.diff(vals)

    # Sign of slope; treat zeros by forward-filling non-zero signs for robustness
    sign = np.sign(d)
    # Replace zeros with previous non-zero sign to avoid spurious flat segments
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]
    if sign[0] == 0:
        # if still zero (completely flat), use +1 arbitrarily
        sign[0] = 1

    # Local extrema: where sign changes
    change = np.where(np.diff(sign) != 0)[0] + 1  # +1 maps to the index in 'vals'
    # Include endpoints as potential extrema
    extrema_idx = np.r_[0, change, len(vals)-1]
    extrema_idx = np.unique(extrema_idx)

    # Compute adjacent peak-to-trough differences
    amp_max = -np.inf
    best_pair = (extrema_idx[0], extrema_idx[1])
    for i in range(len(extrema_idx)-1):
        i1, i2 = extrema_idx[i], extrema_idx[i+1]
        diff = abs(vals[i2] - vals[i1])
        if diff > amp_max:
            amp_max = diff
            best_pair = (i1, i2)

    i1, i2 = best_pair
    t1, t2 = idxs[i1], idxs[i2]
    v1, v2 = vals[i1], vals[i2]
    return 0.5 * amp_max, t1, t2, v1, v2

# --- Compute maximum amplitudes and ratio ---
amp_meas, t1m, t2m, v1m, v2m = max_half_range_amplitude(df["Measured"])
amp_simu, t1s, t2s, v1s, v2s = max_half_range_amplitude(df["Simulated"])
amp_ratio = amp_simu / amp_meas if np.isfinite(amp_meas) and amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_ns = np.median(np.diff(df.index.view("int64")))
dt_sec = float(dt_ns) / 1e9 if (np.isfinite(dt_ns) and dt_ns > 0) else 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals  = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

phase_diff_hours = best_k * dt_hours  # Sim - Obs (positive => Sim lags Obs)

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print("\n-- Maximum half-range amplitudes (adjacent peak↔trough) --")
print(f"Measured amplitude  : {amp_meas:.3f} m  (from {v1m:.3f} at {t1m} to {v2m:.3f} at {t2m})")
print(f"Simulated amplitude : {amp_simu:.3f} m  (from {v1s:.3f} at {t1s} to {v2s:.3f} at {t2s})")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")

print("\n-- Phase --")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2017-08-23 17:35:49  to  2017-08-28 12:35:49

-- Maximum half-range amplitudes (adjacent peak↔trough) --
Measured amplitude  : 0.389 m  (from 2.124 at 2017-08-26 07:18:00 to 1.346 at 2017-08-26 14:12:00)
Simulated amplitude : 1.046 m  (from 0.202 at 2017-08-25 13:36:00 to 2.294 at 2017-08-26 02:42:00)
Amplitude ratio (Sim/Obs): 2.689

-- Phase --
Phase difference (Sim - Obs): 1.70 hours
Max-correlation at lag k=17 samples (corr=0.933); dt = 0.100 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\1641462033.py:104: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [12]:
import numpy as np
import pandas as pd

def max_half_range_amplitude(series: pd.Series, resample_seconds: int | None = None) -> float:
    """
    Maximum half-range amplitude in a time series:
    max over adjacent extrema of |peak - trough| / 2.
    Optionally resample to uniform spacing (linear interp) for robustness.
    """
    s = series.dropna().copy()
    if s.empty:
        return np.nan

    # Optional uniform resample
    if resample_seconds is not None and len(s) > 2:
        freq = f"{int(resample_seconds)}S"
        s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq=freq)).interpolate("time").dropna()

    if len(s) < 3:
        return 0.5 * (s.max() - s.min())

    vals = s.values

    # First diff and sign; handle flats by propagating last nonzero sign
    d = np.diff(vals)
    sign = np.sign(d).astype(int)
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]
    if sign[0] == 0:
        sign[0] = 1  # arbitrary if totally flat start

    # Extrema where sign changes; include endpoints
    change = np.where(np.diff(sign) != 0)[0] + 1
    extrema = np.unique(np.r_[0, change, len(vals) - 1])

    # Max adjacent swing
    max_swing = 0.0
    for i in range(len(extrema) - 1):
        a, b = extrema[i], extrema[i + 1]
        swing = abs(vals[b] - vals[a])
        if swing > max_swing:
            max_swing = swing

    return 0.5 * max_swing

# --- Use with your existing df (time-indexed; columns: "Measured", "Simulated") ---
# Optionally infer a representative dt and resample to uniform spacing
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]
if len(df) >= 2:
    dt_ns = np.median(np.diff(df.index.view("int64")))
    resample_seconds = int(max(1, round((dt_ns / 1e9)))) if np.isfinite(dt_ns) and dt_ns > 0 else None
else:
    resample_seconds = None

amp_meas = max_half_range_amplitude(df["Measured"],  resample_seconds)
amp_simu = max_half_range_amplitude(df["Simulated"], resample_seconds)
amp_ratio = amp_simu / amp_meas if np.isfinite(amp_meas) and amp_meas != 0 else np.nan

print(f"Measured max amplitude  : {amp_meas:.3f} m")
print(f"Simulated max amplitude : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")


Measured max amplitude  : 0.389 m
Simulated max amplitude : 1.046 m
Amplitude ratio (Sim/Obs): 2.689


C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\396837676.py:17: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq=freq)).interpolate("time").dropna()
C:\Users\sahad2\AppData\Local\Temp\ipykernel_35932\396837676.py:17: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq=freq)).interpolate("time").dropna()


In [2]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Aransas wildlife refuge JUL 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\simulated_aransas_rockport_Sbird_Rincon.dfs0"
sim_item_index = 0   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 00:00:00")
t_end   = pd.to_datetime("2022-07-31 00:00:00")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 1 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# ---------- Helper: maximum amplitude (largest adjacent peak↔trough swing / 2) ----------
def max_half_range_amplitude(series: pd.Series):
    """
    Returns:
        amp_max: maximum half-range amplitude (float)
        t1, t2 : timestamps of the trough/peak pair defining amp_max (pd.Timestamp)
        v1, v2 : values at those timestamps (float, float)
    Notes:
        Finds local extrema (including endpoints) and computes the maximum
        adjacent peak-to-trough half-range within the window.
    """
    s = series.dropna()
    if len(s) < 3:
        # Fallback: use global half-range
        vmin, vmax = s.min(), s.max()
        return 0.5 * (vmax - vmin), s.idxmin(), s.idxmax(), vmin, vmax

    vals = s.values
    idxs = s.index

    # First differences
    d = np.diff(vals)

    # Sign of slope; treat zeros by forward-filling non-zero signs for robustness
    sign = np.sign(d)
    # Replace zeros with previous non-zero sign to avoid spurious flat segments
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]
    if sign[0] == 0:
        # if still zero (completely flat), use +1 arbitrarily
        sign[0] = 1

    # Local extrema: where sign changes
    change = np.where(np.diff(sign) != 0)[0] + 1  # +1 maps to the index in 'vals'
    # Include endpoints as potential extrema
    extrema_idx = np.r_[0, change, len(vals)-1]
    extrema_idx = np.unique(extrema_idx)

    # Compute adjacent peak-to-trough differences
    amp_max = -np.inf
    best_pair = (extrema_idx[0], extrema_idx[1])
    for i in range(len(extrema_idx)-1):
        i1, i2 = extrema_idx[i], extrema_idx[i+1]
        diff = abs(vals[i2] - vals[i1])
        if diff > amp_max:
            amp_max = diff
            best_pair = (i1, i2)

    i1, i2 = best_pair
    t1, t2 = idxs[i1], idxs[i2]
    v1, v2 = vals[i1], vals[i2]
    return 0.5 * amp_max, t1, t2, v1, v2

# --- Compute maximum amplitudes and ratio ---
amp_meas, t1m, t2m, v1m, v2m = max_half_range_amplitude(df["Measured"])
amp_simu, t1s, t2s, v1s, v2s = max_half_range_amplitude(df["Simulated"])
amp_ratio = amp_simu / amp_meas if np.isfinite(amp_meas) and amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_ns = np.median(np.diff(df.index.view("int64")))
dt_sec = float(dt_ns) / 1e9 if (np.isfinite(dt_ns) and dt_ns > 0) else 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals  = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

phase_diff_hours = best_k * dt_hours  # Sim - Obs (positive => Sim lags Obs)

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print("\n-- Maximum half-range amplitudes (adjacent peak↔trough) --")
print(f"Measured amplitude  : {amp_meas:.3f} m  (from {v1m:.3f} at {t1m} to {v2m:.3f} at {t2m})")
print(f"Simulated amplitude : {amp_simu:.3f} m  (from {v1s:.3f} at {t1s} to {v2s:.3f} at {t2s})")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")

print("\n-- Phase --")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2022-07-06 00:00:00  to  2022-07-31 00:00:00

-- Maximum half-range amplitudes (adjacent peak↔trough) --
Measured amplitude  : 0.068 m  (from -0.101 at 2022-07-15 13:30:00 to 0.035 at 2022-07-15 15:30:00)
Simulated amplitude : 0.126 m  (from -0.099 at 2022-07-13 09:00:00 to 0.153 at 2022-07-13 21:00:00)
Amplitude ratio (Sim/Obs): 1.850

-- Phase --
Phase difference (Sim - Obs): 27.00 hours
Max-correlation at lag k=54 samples (corr=0.389); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_57976\216543061.py:104: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [3]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Aransas wildlife refuge JUL 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\simulated_aransas_rockport_Sbird_Rincon.dfs0"
sim_item_index = 0   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 00:00:00")
t_end   = pd.to_datetime("2022-07-31 00:00:00")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 1 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# ---------- Helper: maximum amplitude (largest adjacent peak↔trough swing / 2) ----------
def max_half_range_amplitude(series: pd.Series):
    """
    Returns:
        amp_max: maximum half-range amplitude (float)
        t1, t2 : timestamps of the trough/peak pair defining amp_max (pd.Timestamp)
        v1, v2 : values at those timestamps (float, float)
    Notes:
        Finds local extrema (including endpoints) and computes the maximum
        adjacent peak-to-trough half-range within the window.
    """
    s = series.dropna()
    if len(s) < 3:
        # Fallback: use global half-range
        vmin, vmax = s.min(), s.max()
        return 0.5 * (vmax - vmin), s.idxmin(), s.idxmax(), vmin, vmax

    vals = s.values
    idxs = s.index

    # First differences
    d = np.diff(vals)

    # Sign of slope; treat zeros by forward-filling non-zero signs for robustness
    sign = np.sign(d)
    # Replace zeros with previous non-zero sign to avoid spurious flat segments
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]
    if sign[0] == 0:
        # if still zero (completely flat), use +1 arbitrarily
        sign[0] = 1

    # Local extrema: where sign changes
    change = np.where(np.diff(sign) != 0)[0] + 1  # +1 maps to the index in 'vals'
    # Include endpoints as potential extrema
    extrema_idx = np.r_[0, change, len(vals)-1]
    extrema_idx = np.unique(extrema_idx)

    # Compute adjacent peak-to-trough differences
    amp_max = -np.inf
    best_pair = (extrema_idx[0], extrema_idx[1])
    for i in range(len(extrema_idx)-1):
        i1, i2 = extrema_idx[i], extrema_idx[i+1]
        diff = abs(vals[i2] - vals[i1])
        if diff > amp_max:
            amp_max = diff
            best_pair = (i1, i2)

    i1, i2 = best_pair
    t1, t2 = idxs[i1], idxs[i2]
    v1, v2 = vals[i1], vals[i2]
    return 0.5 * amp_max, t1, t2, v1, v2

# --- Compute maximum amplitudes and ratio ---
amp_meas, t1m, t2m, v1m, v2m = max_half_range_amplitude(df["Measured"])
amp_simu, t1s, t2s, v1s, v2s = max_half_range_amplitude(df["Simulated"])
amp_ratio = amp_simu / amp_meas if np.isfinite(amp_meas) and amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_ns = np.median(np.diff(df.index.view("int64")))
dt_sec = float(dt_ns) / 1e9 if (np.isfinite(dt_ns) and dt_ns > 0) else 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals  = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

phase_diff_hours = best_k * dt_hours  # Sim - Obs (positive => Sim lags Obs)

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print("\n-- Maximum half-range amplitudes (adjacent peak↔trough) --")
print(f"Measured amplitude  : {amp_meas:.3f} m  (from {v1m:.3f} at {t1m} to {v2m:.3f} at {t2m})")
print(f"Simulated amplitude : {amp_simu:.3f} m  (from {v1s:.3f} at {t1s} to {v2s:.3f} at {t2s})")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")

print("\n-- Phase --")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2022-07-06 00:00:00  to  2022-07-31 00:00:00

-- Maximum half-range amplitudes (adjacent peak↔trough) --
Measured amplitude  : 0.068 m  (from -0.101 at 2022-07-15 13:30:00 to 0.035 at 2022-07-15 15:30:00)
Simulated amplitude : 0.126 m  (from -0.099 at 2022-07-13 09:00:00 to 0.153 at 2022-07-13 21:00:00)
Amplitude ratio (Sim/Obs): 1.850

-- Phase --
Phase difference (Sim - Obs): 27.00 hours
Max-correlation at lag k=54 samples (corr=0.389); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_57976\216543061.py:104: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [4]:
import numpy as np
import pandas as pd
from mikeio import Dfs0

# ---------- USER INPUT ----------
measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\Rock Port July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\simulated_aransas_rockport_Sbird_Rincon.dfs0"
sim_item_index = 1   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 00:00:00")
t_end   = pd.to_datetime("2022-07-31 00:00:00")
# --------------------------------

# --- Read dfs0 files ---
meas = Dfs0(measured_path).read()                         # full file
simu = Dfs0(simulated_path).read(items=[sim_item_index])  # only Item 1 (0-based index=1)

# --- Convert to Pandas and align ---
meas_df = meas.to_dataframe()
simu_df = simu.to_dataframe()

# Keep only the analysis window
meas_df = meas_df.loc[t_start:t_end]
simu_df = simu_df.loc[t_start:t_end]

# Inner-join on timestamps and drop NaNs
df = pd.concat([meas_df, simu_df], axis=1, join="inner").dropna()
df.columns = ["Measured", "Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# ---------- Helper: maximum amplitude (largest adjacent peak↔trough swing / 2) ----------
def max_half_range_amplitude(series: pd.Series):
    """
    Returns:
        amp_max: maximum half-range amplitude (float)
        t1, t2 : timestamps of the trough/peak pair defining amp_max (pd.Timestamp)
        v1, v2 : values at those timestamps (float, float)
    Notes:
        Finds local extrema (including endpoints) and computes the maximum
        adjacent peak-to-trough half-range within the window.
    """
    s = series.dropna()
    if len(s) < 3:
        # Fallback: use global half-range
        vmin, vmax = s.min(), s.max()
        return 0.5 * (vmax - vmin), s.idxmin(), s.idxmax(), vmin, vmax

    vals = s.values
    idxs = s.index

    # First differences
    d = np.diff(vals)

    # Sign of slope; treat zeros by forward-filling non-zero signs for robustness
    sign = np.sign(d)
    # Replace zeros with previous non-zero sign to avoid spurious flat segments
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]
    if sign[0] == 0:
        # if still zero (completely flat), use +1 arbitrarily
        sign[0] = 1

    # Local extrema: where sign changes
    change = np.where(np.diff(sign) != 0)[0] + 1  # +1 maps to the index in 'vals'
    # Include endpoints as potential extrema
    extrema_idx = np.r_[0, change, len(vals)-1]
    extrema_idx = np.unique(extrema_idx)

    # Compute adjacent peak-to-trough differences
    amp_max = -np.inf
    best_pair = (extrema_idx[0], extrema_idx[1])
    for i in range(len(extrema_idx)-1):
        i1, i2 = extrema_idx[i], extrema_idx[i+1]
        diff = abs(vals[i2] - vals[i1])
        if diff > amp_max:
            amp_max = diff
            best_pair = (i1, i2)

    i1, i2 = best_pair
    t1, t2 = idxs[i1], idxs[i2]
    v1, v2 = vals[i1], vals[i2]
    return 0.5 * amp_max, t1, t2, v1, v2

# --- Compute maximum amplitudes and ratio ---
amp_meas, t1m, t2m, v1m, v2m = max_half_range_amplitude(df["Measured"])
amp_simu, t1s, t2s, v1s, v2s = max_half_range_amplitude(df["Simulated"])
amp_ratio = amp_simu / amp_meas if np.isfinite(amp_meas) and amp_meas != 0 else np.nan

# --- Phase difference as time lag (Sim - Obs), via correlation-based lag search ---
# Ensure (approximately) uniform sampling interval:
if df.index.nunique() != len(df):
    df = df[~df.index.duplicated(keep="first")]

# Infer representative timestep (seconds) from median delta
dt_ns = np.median(np.diff(df.index.view("int64")))
dt_sec = float(dt_ns) / 1e9 if (np.isfinite(dt_ns) and dt_ns > 0) else 600.0

# Optionally resample to a uniform grid for robustness (keeps values via linear interpolation)
freq_str = f"{int(round(dt_sec))}S"
idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)
df_u = df.reindex(idx_uniform).interpolate(method="time").dropna()

# Search lags within ±48 hours
max_hours = 48
dt_hours = dt_sec / 3600.0
max_shift = int(round(max_hours / dt_hours))

meas_vals = df_u["Measured"]
sim_vals  = df_u["Simulated"]

best_k = 0
best_corr = -np.inf
for k in range(-max_shift, max_shift + 1):
    corr = meas_vals.corr(sim_vals.shift(k))
    if pd.notna(corr) and corr > best_corr:
        best_corr = corr
        best_k = k

phase_diff_hours = best_k * dt_hours  # Sim - Obs (positive => Sim lags Obs)

# --- Print results ---
print("=== Amplitude and Phase Analysis (Windowed) ===")
print(f"Time window: {t_start}  to  {t_end}")
print("\n-- Maximum half-range amplitudes (adjacent peak↔trough) --")
print(f"Measured amplitude  : {amp_meas:.3f} m  (from {v1m:.3f} at {t1m} to {v2m:.3f} at {t2m})")
print(f"Simulated amplitude : {amp_simu:.3f} m  (from {v1s:.3f} at {t1s} to {v2s:.3f} at {t2s})")
print(f"Amplitude ratio (Sim/Obs): {amp_ratio:.3f}")

print("\n-- Phase --")
print(f"Phase difference (Sim - Obs): {phase_diff_hours:.2f} hours")
print(f"Max-correlation at lag k={best_k} samples (corr={best_corr:.3f}); dt = {dt_hours:.3f} h/sample")


=== Amplitude and Phase Analysis (Windowed) ===
Time window: 2022-07-06 00:00:00  to  2022-07-31 00:00:00

-- Maximum half-range amplitudes (adjacent peak↔trough) --
Measured amplitude  : 0.093 m  (from -0.163 at 2022-07-11 03:30:00 to 0.022 at 2022-07-11 16:00:00)
Simulated amplitude : 0.089 m  (from -0.065 at 2022-07-13 06:00:00 to 0.113 at 2022-07-13 19:30:00)
Amplitude ratio (Sim/Obs): 0.961

-- Phase --
Phase difference (Sim - Obs): 25.00 hours
Max-correlation at lag k=50 samples (corr=0.583); dt = 0.500 h/sample


C:\Users\sahad2\AppData\Local\Temp\ipykernel_57976\1721456883.py:104: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  idx_uniform = pd.date_range(df.index[0], df.index[-1], freq=freq_str)


In [7]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\Rock Port July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\simulated_aransas_rockport_Sbird_Rincon.dfs0"
sim_item_index = 1   # Item no. 2 (0-based indexing)

# Analysis window (inclusive)
t_start=pd.to_datetime("2022-07-06 00:00:00")
t_end=pd.to_datetime("2022-07-31 00:00:00")

# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2022-07-06 00:00:00 to 2022-07-31 00:00:00
Measured amplitude          : 0.152 m
Simulated amplitude         : 0.091 m
Amplitude ratio (Sim/Obs)   : 0.597
Number of matched peaks     : 28
Mean phase difference       : -0.11 hours (Sim - Obs)
Mean absolute phase error   : 0.96 hours
Std. of phase differences   : 1.24 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2022-07-06 01:30:00 2022-07-06 00:30:00                    -1.0
2022-07-06 15:30:00 2022-07-06 18:30:00                     3.0
2022-07-07 04:30:00 2022-07-07 05:00:00                     0.5
2022-07-07 16:00:00 2022-07-07 16:30:00                     0.5
2022-07-08 16:00:00 2022-07-08 15:30:00                    -0.5
2022-07-09 16:00:00 2022-07-09 16:00:00                     0.0
2022-07-10 16:00:00 2022-07-10 16:30:00                     0.5
2022-07-11 16:00:00 2022-07-11 17:30:00             

In [8]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Aransas wildlife refuge JUL 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\simulated_aransas_rockport_Sbird_Rincon.dfs0"
sim_item_index = 0   # Item no. 1 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 00:00:00")
t_end   = pd.to_datetime("2022-07-31 00:00:00")

# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2022-07-06 00:00:00 to 2022-07-31 00:00:00
Measured amplitude          : 0.161 m
Simulated amplitude         : 0.127 m
Amplitude ratio (Sim/Obs)   : 0.787
Number of matched peaks     : 24
Mean phase difference       : -0.35 hours (Sim - Obs)
Mean absolute phase error   : 2.90 hours
Std. of phase differences   : 3.27 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2022-07-06 22:00:00 2022-07-06 21:30:00                    -0.5
2022-07-07 23:00:00 2022-07-07 19:30:00                    -3.5
2022-07-08 20:30:00 2022-07-08 16:00:00                    -4.5
2022-07-09 12:30:00 2022-07-09 17:00:00                     4.5
2022-07-10 19:30:00 2022-07-10 18:00:00                    -1.5
2022-07-11 22:00:00 2022-07-11 19:00:00                    -3.0
2022-07-13 00:00:00 2022-07-12 20:00:00                    -4.0
2022-07-14 00:30:00 2022-07-13 21:00:00             

In [9]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Port Lavaca Jul 2023.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_July_2023.m21fm - Result Files\Simulated_Aransas.dfs0"
sim_item_index = 2   # Item no. 3 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2023-07-01 02:26:50")
t_end   = pd.to_datetime("2023-07-16 19:46:27")

# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2023-07-01 02:26:50 to 2023-07-16 19:46:27
Measured amplitude          : 0.259 m
Simulated amplitude         : 0.325 m
Amplitude ratio (Sim/Obs)   : 1.253
Number of matched peaks     : 16
Mean phase difference       : -1.53 hours (Sim - Obs)
Mean absolute phase error   : 3.03 hours
Std. of phase differences   : 3.15 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2023-07-02 15:00:00 2023-07-02 12:00:00                    -3.0
2023-07-03 15:00:00 2023-07-03 11:30:00                    -3.5
2023-07-04 17:00:00 2023-07-04 12:30:00                    -4.5
2023-07-05 18:00:00 2023-07-05 13:00:00                    -5.0
2023-07-06 14:30:00 2023-07-06 13:30:00                    -1.0
2023-07-07 08:00:00 2023-07-07 14:00:00                     6.0
2023-07-08 07:30:00 2023-07-08 13:30:00                     6.0
2023-07-09 14:30:00 2023-07-09 13:00:00             

In [10]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Prototype HD Model\Measured WL\Portocornor JUL 2023.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\test_run_July_2023.m21fm - Result Files\Simulated_Aransas.dfs0"
sim_item_index = 1   # Item no. 2 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2023-07-01 13:35:32")
t_end   = pd.to_datetime("2023-07-16 12:03:10")
# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2023-07-01 13:35:32 to 2023-07-16 12:03:10
Measured amplitude          : 0.238 m
Simulated amplitude         : 0.278 m
Amplitude ratio (Sim/Obs)   : 1.165
Number of matched peaks     : 17
Mean phase difference       : -0.65 hours (Sim - Obs)
Mean absolute phase error   : 1.65 hours
Std. of phase differences   : 2.24 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2023-07-02 00:00:00 2023-07-02 02:30:00                     2.5
2023-07-02 11:00:00 2023-07-02 10:30:00                    -0.5
2023-07-03 12:00:00 2023-07-03 11:00:00                    -1.0
2023-07-04 14:00:00 2023-07-04 12:00:00                    -2.0
2023-07-05 14:30:00 2023-07-05 13:00:00                    -1.5
2023-07-06 15:30:00 2023-07-06 14:00:00                    -1.5
2023-07-07 08:00:00 2023-07-07 14:00:00                     6.0
2023-07-08 15:00:00 2023-07-08 13:00:00             

In [11]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\La quinta channel July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\Simulated_Matagorda v2.dfs0"
sim_item_index = 7   # Item no. 8 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-08 1:16:20")
t_end   = pd.to_datetime("2022-07-29 20:24:38")
# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2022-07-08 01:16:20 to 2022-07-29 20:24:38
Measured amplitude          : 0.223 m
Simulated amplitude         : 0.199 m
Amplitude ratio (Sim/Obs)   : 0.894
Number of matched peaks     : 24
Mean phase difference       : -0.98 hours (Sim - Obs)
Mean absolute phase error   : 2.98 hours
Std. of phase differences   : 3.33 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2022-07-08 10:00:00 2022-07-08 09:30:00                    -0.5
2022-07-09 17:00:00 2022-07-09 11:30:00                    -5.5
2022-07-10 14:00:00 2022-07-10 12:30:00                    -1.5
2022-07-11 16:30:00 2022-07-11 14:00:00                    -2.5
2022-07-12 09:00:00 2022-07-12 15:00:00                     6.0
2022-07-13 20:00:00 2022-07-13 16:30:00                    -3.5
2022-07-14 20:30:00 2022-07-14 17:30:00                    -3.0
2022-07-15 20:00:00 2022-07-15 18:30:00             

In [12]:
import numpy as np
import pandas as pd
from mikeio import Dfs0
from scipy.signal import find_peaks

# ---------- USER INPUT ----------

measured_path  = r"D:\Phd Research\Data\Water Level Data\Mike time series dfso\USS Lexington July 2022.dfs0"
simulated_path = r"D:\Phd Research\Prototype HD Model\Simulation\July_2022.m21fm - Result Files\Simulated_Matagorda v2.dfs0"
sim_item_index = 6   # Item no. 7 (0-based indexing)

# Analysis window (inclusive)
t_start = pd.to_datetime("2022-07-06 12:43:18")
t_end   = pd.to_datetime("2022-07-31 9:26:50")
# Peak detection/matching settings
MIN_PEAK_DISTANCE_HOURS=8       # prevents multiple small peaks within one tidal cycle
MATCH_WINDOW_HOURS=6            # max allowed Obs-Sim peak separation

# --------------------------------

# --- Read dfs0 files ---
meas=Dfs0(measured_path).read()
simu=Dfs0(simulated_path).read(items=[sim_item_index])

# --- Convert to Pandas and align ---
meas_df=meas.to_dataframe().loc[t_start:t_end]
simu_df=simu.to_dataframe().loc[t_start:t_end]

df=pd.concat([meas_df,simu_df],axis=1,join="inner").dropna()
df.columns=["Measured","Simulated"]

if df.empty:
    raise RuntimeError("No overlapping data in the specified time window after alignment.")

# Remove duplicate timestamps
df=df[~df.index.duplicated(keep="first")]

# ============================================================
# AMPLITUDE
# ============================================================

amp_meas=0.5*(df["Measured"].max()-df["Measured"].min())
amp_simu=0.5*(df["Simulated"].max()-df["Simulated"].min())
amp_ratio=amp_simu/amp_meas if amp_meas!=0 else np.nan

# ============================================================
# PEAK-BASED PHASE DIFFERENCE
# ============================================================

# Representative timestep
dt_sec=np.median(np.diff(df.index.view("int64")))/1e9
dt_hours=dt_sec/3600
min_peak_samples=max(1,int(round(MIN_PEAK_DISTANCE_HOURS/dt_hours)))

# Detect observed and simulated high-water peaks
obs_idx,_=find_peaks(df["Measured"].values,distance=min_peak_samples)
sim_idx,_=find_peaks(df["Simulated"].values,distance=min_peak_samples)

obs_times=df.index[obs_idx]
sim_times=df.index[sim_idx]

# Match each observed peak to nearest unused simulated peak
matched=[]
used_sim=set()

for obs_time in obs_times:
    candidates=[(j,sim_time) for j,sim_time in enumerate(sim_times)
                if j not in used_sim and abs((sim_time-obs_time).total_seconds()/3600)<=MATCH_WINDOW_HOURS]

    if candidates:
        j,sim_time=min(candidates,key=lambda x:abs((x[1]-obs_time).total_seconds()))
        phase_hours=(sim_time-obs_time).total_seconds()/3600
        matched.append([obs_time,sim_time,phase_hours])
        used_sim.add(j)

# Results table
phase_df=pd.DataFrame(matched,columns=["Observed_peak","Simulated_peak","Phase_difference_hours"])

if phase_df.empty:
    raise RuntimeError("No corresponding observed/simulated peaks found within the matching window.")

# Mean signed phase difference
mean_phase=phase_df["Phase_difference_hours"].mean()

# Mean absolute peak timing error
mean_abs_phase=phase_df["Phase_difference_hours"].abs().mean()

# Standard deviation
std_phase=phase_df["Phase_difference_hours"].std()

# ============================================================
# RESULTS
# ============================================================

print("\n=== Amplitude and Peak-Based Phase Analysis ===")
print(f"Time window                 : {t_start} to {t_end}")
print(f"Measured amplitude          : {amp_meas:.3f} m")
print(f"Simulated amplitude         : {amp_simu:.3f} m")
print(f"Amplitude ratio (Sim/Obs)   : {amp_ratio:.3f}")
print(f"Number of matched peaks     : {len(phase_df)}")
print(f"Mean phase difference       : {mean_phase:.2f} hours (Sim - Obs)")
print(f"Mean absolute phase error   : {mean_abs_phase:.2f} hours")
print(f"Std. of phase differences   : {std_phase:.2f} hours")

print("\n=== Individual Peak Matches ===")
print(phase_df.to_string(index=False))


=== Amplitude and Peak-Based Phase Analysis ===
Time window                 : 2022-07-06 12:43:18 to 2022-07-31 09:26:50
Measured amplitude          : 0.242 m
Simulated amplitude         : 0.205 m
Amplitude ratio (Sim/Obs)   : 0.847
Number of matched peaks     : 27
Mean phase difference       : -1.83 hours (Sim - Obs)
Mean absolute phase error   : 3.65 hours
Std. of phase differences   : 3.61 hours

=== Individual Peak Matches ===
      Observed_peak      Simulated_peak  Phase_difference_hours
2022-07-06 18:30:00 2022-07-06 17:30:00                    -1.0
2022-07-07 06:00:00 2022-07-07 07:00:00                     1.0
2022-07-07 21:00:00 2022-07-07 17:30:00                    -3.5
2022-07-09 17:00:00 2022-07-09 11:30:00                    -5.5
2022-07-10 07:30:00 2022-07-10 13:00:00                     5.5
2022-07-11 17:00:00 2022-07-11 14:00:00                    -3.0
2022-07-12 18:30:00 2022-07-12 15:30:00                    -3.0
2022-07-13 21:30:00 2022-07-13 16:30:00             